## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity

from data_processing import processing as proc
from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing import types as proc_types
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

## Experiment ID Input

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
detector_code = Detector.ONE

In [ ]:
# default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )

# fit_styles: dict[int, SliceFitStyle] = {
#     1: "bounds",
#     2: "peak_finder"
# }
# fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])
fit_style = "peak_finder"

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
# bin_length = helpers.get_input_with_default(
#     "Enter bin length (in seconds), or press Enter for default (300 s)",
#     300,
#     int
# )
bin_length = 300
bin_string = f"{bin_length}s"

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
    'bin_length': bin_length
}

In [ ]:
analysis_timestamp

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# # Generate histogram

# start_scan_idx = 0
# end_scan_idx = 420
# energy_width = 5e-3
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width

# for exp_id, exp_data in experiment_neutron_data.items():
#     psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
#     Z, xe, ye = get_psd_energy_histogram(
#         psd_report,
#         calibrated_energy_column,
#         energy_width=energy_width
#     )
#     exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
#     exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
#     exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
#     exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
adc_width = 20
# adc_width = 100
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_width=adc_width,
        # psd_bin_count=50
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    if fit_style == "bounds":
        # Default
        default_bounds: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
        )

        bounds_a: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
        )

        bounds_b: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
        )

        # Ranged Example
        bounds = [
            ((0, 60), bounds_a),
        ]
    else:
        default_bounds = None
        bounds = None

    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style=fit_style,
        default_bounds=default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, exp_id, nan_total_threshold=10)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
print(experiment_neutron_data["TB-26"][ExperimentDataKey.BORDERS])

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Get experiment start time
for exp_name, data_dict in experiment_neutron_data.items():
    exp_root = get_exp_root(exp_name)
    with open(exp_root / 'exp_info.toml') as exp_info:
        exp_start_line = [line for line in exp_info if "exp_start" in line][0]
    exp_start_text = exp_start_line.replace("exp_start = ", "").strip()
    exp_start = datetime.fromisoformat(exp_start_text).astimezone(timezone.utc)
    data_dict[ExperimentDataKey.START_TIME] = exp_start

In [ ]:
# Get timetag as clock time
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    exp_start = data_dict[ExperimentDataKey.START_TIME]

    psd_report = calculate_event_time(psd_report, exp_start)

    data_dict[ExperimentDataKey.PSD_REPORT] = psd_report

In [ ]:
# Separate neutron and gamma events
for exp_name, data_dict in experiment_neutron_data.items():
    psd_report = data_dict[ExperimentDataKey.PSD_REPORT]

    n_classify_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    neutrons_only = psd_report.query(n_classify_col_name).copy()
    gamma_only = psd_report.query(f"~{n_classify_col_name}").copy()
    data_dict[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    data_dict[ExperimentDataKey.GAMMA_ONLY] = gamma_only

### Non-Neutron Data Processing

In [ ]:
# # process reactor data files
# # stored in reactor_data
# # File name format: Device Param 00x
# # If same device/param, but different numbers, should be merged

# data_file_pattern = re.compile(r"([a-zA-Z ]+) (\d+)")
# for exp_name, data_dict in experiment_neutron_data.items():
#     reactor_data_folder = get_reactor_data_root(exp_name)
#     time_col_name = NonReactorDataframeColumn.TIME.value
#     data_col_name = NonReactorDataframeColumn.DATA.value
#     units_col_name = NonReactorDataframeColumn.UNITS.value
    
#     if not reactor_data_folder.is_dir():
#         continue

#     reactor_data_files = defaultdict(list)
#     for file in reactor_data_folder.iterdir():
#         match = data_file_pattern.match(file.name)
#         if match and len(match.groups()) > 0:
#             data_source = match.group(1)
#             if isinstance(data_source, str):
#                 reactor_data_files[data_source].append(file)

#     reactor_data = {}
#     for data_source, files in reactor_data_files.items():
#         file_dfs = [
#             pd.read_csv(
#                 file,
#                 header=0,
#                 dtype=str,
#                 encoding='cp1252',
#                 names=[time_col_name, data_col_name, units_col_name]
#             ) for file in sorted(files)]
#         for df in file_dfs:
#             df[time_col_name] = pd.to_datetime(
#                 df[time_col_name], utc=True)
#         df = pd.concat(file_dfs, ignore_index=True)
#         reactor_data[data_source] = df

#     data_dict[ExperimentDataKey.REACTOR_DATA] = reactor_data

## Data Binning

In [ ]:
# Create bins
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]

    event_time_col = DetectorDataframeColumn.EVENT_TIME.value
    start_time = neutron_report[event_time_col].min()
    end_time = neutron_report[event_time_col].max()
    timetag_clock_bins = pd.date_range(
        start=start_time, end=end_time, freq=bin_string)
    data_dict[ExperimentDataKey.TIME_BIN_EDGES] = timetag_clock_bins

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    time_bin_edges = exp_data[ExperimentDataKey.TIME_BIN_EDGES]
    bin_index = 0
    low_time_edge = time_bin_edges[bin_index]
    high_time_edge = time_bin_edges[bin_index + 1]
    event_time_col = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_df = psd_report[
        psd_report[event_time_col].between(low_time_edge, high_time_edge)
    ]

    Z, xe, ye = get_psd_adc_histogram(
        time_bin_df,
        adc_width=adc_width
    )
    exp_data["time_bin_histogram"] = Z
    exp_data["time_bin_energy_edges"] = xe
    exp_data["time_bin_psd_edges"] = ye
    # Z, xe, ye = get_psd_adc_histogram(
    #     psd_report,
    #     adc_width=adc_width,
    #     # psd_bin_count=50
    # )
    # exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    # exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    # exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye

In [ ]:
# Bin neutron data
for exp_name, data_dict in experiment_neutron_data.items():
    neutron_report = data_dict[ExperimentDataKey.NEUTRONS_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value

    start_time = time_bins[0]

    binned_neutrons = get_time_cut(
        neutron_report, time_col_name, time_bins)
    binned_neutrons = neutron_report.groupby(
        time_bin_col_name, as_index=True, observed=False) \
        .size() \
        .to_frame() \
        .copy()
    binned_neutrons.columns = [count_col_name]
    binned_neutrons[count_error_col_name] = np.sqrt(
        binned_neutrons[count_col_name]
    )

    binned_neutron_time_bins = binned_neutrons.index.to_series()
    midpoints = binned_neutron_time_bins.apply(lambda x: x.mid)
    durations = binned_neutron_time_bins.apply(
        lambda x: x.length.total_seconds()
    ).astype(np.float64)

    binned_neutrons[bin_mid_col_name] = midpoints
    binned_neutrons = bin_midpoint_time_to_seconds(binned_neutrons, start_time)

    binned_neutrons[n_rate_col_name] = (
        binned_neutrons[count_col_name] / durations)
    binned_neutrons[n_error_col_name] = (
        binned_neutrons[count_error_col_name] / durations)
    binned_neutrons = binned_neutrons.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ) \
        .copy()
    data_dict[ExperimentDataKey.BINNED_NEUTRONS] = binned_neutrons

In [ ]:
# Bin gamma data
for exp_name, data_dict in experiment_neutron_data.items():
    gamma_report = data_dict[ExperimentDataKey.GAMMA_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value
    count_error_col_name = BinningDataframeColumn.COUNT_ERROR.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    g_rate_col_name = BinningDataframeColumn.GAMMA_RATE.value
    g_error_col_name = BinningDataframeColumn.GAMMA_RATE_ERROR.value

    start_time = time_bins[0]

    binned_gamma = get_time_cut(
        gamma_report, time_col_name, time_bins)
    binned_gamma = gamma_report.groupby(
        time_bin_col_name, as_index=True, observed=True) \
        .size() \
        .to_frame() \
        .copy()
    binned_gamma.columns = [count_col_name]
    binned_gamma[count_error_col_name] = np.sqrt(
        binned_gamma[count_col_name])

    binned_gamma_time_bins = binned_gamma.index.to_series()
    midpoints = binned_gamma_time_bins.apply(lambda x: x.mid)
    durations = binned_gamma_time_bins.apply(
        lambda x: x.length.total_seconds()).astype(np.float64)

    binned_gamma[bin_mid_col_name] = midpoints
    binned_gamma = bin_midpoint_time_to_seconds(binned_gamma, start_time)

    binned_gamma[g_rate_col_name] = (
        binned_gamma[count_col_name] / durations)
    binned_gamma[g_error_col_name] = (
        binned_gamma[count_error_col_name] / durations)
    binned_gamma = binned_gamma.drop(
        [count_col_name, count_error_col_name],
        axis=1
    ).copy()
    data_dict[ExperimentDataKey.BINNED_GAMMA] = binned_gamma

In [ ]:
# bin gamma energy spectrum

def make_index_converter(
    start_time: pd.Timestamp
) -> Callable:
    def index_converter(
        category: pd.Interval
    ) -> pd.Interval:
        cat_start = category.left
        cat_end = category.right
        start_seconds = (cat_start - start_time).total_seconds()
        end_seconds = (cat_end - start_time).total_seconds()
        return pd.Interval(start_seconds, end_seconds, closed=category.closed)

    return index_converter


for exp_name, data_dict in experiment_neutron_data.items():
    gamma_report = data_dict[ExperimentDataKey.GAMMA_ONLY]
    time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
    energy_bins = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]

    time_col_name = DetectorDataframeColumn.EVENT_TIME.value
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    eng_bin_col_name = BinningDataframeColumn.ENERGY_BIN.value
    count_col_name = BinningDataframeColumn.COUNT.value

    start_time = time_bins[0]

    binned_gamma_spectrum = get_time_cut(
        gamma_report, time_col_name, time_bins)
    energy_cut, energy_bins = pd.cut(
        binned_gamma_spectrum[calibrated_energy_column.value],
        bins=energy_bins,
        retbins=True
    )
    binned_gamma_spectrum[eng_bin_col_name] = energy_cut
    binned_gamma_spectrum = binned_gamma_spectrum.groupby(
        [time_bin_col_name, eng_bin_col_name],
        as_index=True,
        observed=False
    ) \
        .size() \
        .to_frame() \
        .copy()
    binned_gamma_spectrum.columns = [count_col_name]
    binned_gamma_spectrum = binned_gamma_spectrum.reset_index(level=1)
    binned_gamma_spectrum = binned_gamma_spectrum.pivot_table(
        values=count_col_name,
        index=binned_gamma_spectrum.index,
        columns=eng_bin_col_name,
        observed=False
    )
    
    time_index = binned_gamma_spectrum.index
    start_time = time_index[0].left
    convert_categories = make_index_converter(start_time)
    time_index = time_index.map(convert_categories)
    binned_gamma_spectrum.index = time_index

    data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM] = binned_gamma_spectrum
    data_dict[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES] = energy_bins

In [ ]:
# # process and bin reactor data

# def normalize_units(value, unit, to_unit):
#     '''
#     Converts Series of values to desired units

#     value: measured value
#     units: units of measured value
#     to_unit: unit to convert to

#     returns Series of values converted to desired unit
#     '''
#     try:
#         return Quantity(value, unit).ito(to_unit).magnitude
#     except AttributeError:
#         return value


# for exp_name, data_dict in experiment_neutron_data.items():
#     time_col_name = NonReactorDataframeColumn.TIME.value
#     data_col_name = NonReactorDataframeColumn.DATA.value
#     units_col_name = NonReactorDataframeColumn.UNITS.value
#     norm_data_col_name = NonReactorDataframeColumn.NORMALIZED_DATA.value
#     norm_units_col_name = NonReactorDataframeColumn.NORMALIZED_UNITS.value

#     if (
#         reactor_data := data_dict.get(ExperimentDataKey.REACTOR_DATA)
#     ) is not None:
#         binned_reactor_data = {}
#         time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]

#         for data_source, reactor_param_df in reactor_data.items():
#             try:
#                 reactor_param_df[data_col_name] = reactor_param_df[
#                     data_col_name
#                 ].astype(float)
#             except ValueError:
#                 continue  # skip if not numeric

#             units_counts = reactor_param_df[units_col_name].value_counts()
#             main_unit = units_counts.idxmax()
#             if units_counts.size > 1:
#                 # determine most frequent
#                 reactor_param_df[norm_data_col_name] = reactor_param_df.apply(
#                     lambda row: normalize_units(
#                         row[data_col_name],
#                         row[units_col_name],
#                         main_unit
#                     ),
#                     axis=1
#                 )
#                 reactor_param_df = reactor_param_df.assign(
#                     **{norm_units_col_name: lambda _: main_unit}
#                 )
#             else:
#                 reactor_param_df[norm_data_col_name] = reactor_param_df[
#                     data_col_name]
#                 reactor_param_df[norm_units_col_name] = reactor_param_df[
#                     units_col_name]

#             binned_reactor_param_df = bin_non_neutron_data(
#                 reactor_param_df,
#                 time_bins,
#                 norm_data_col_name,
#                 [f"Average {data_source} ({main_unit})",
#                  f"{data_source} error ({main_unit})"]
#             )
#             binned_reactor_data[data_source] = binned_reactor_param_df
#         data_dict[ExperimentDataKey.BINNED_REACTOR_DATA] = binned_reactor_data

In [ ]:
# Merge binned data
for exp_name, data_dict in experiment_neutron_data.items():
    time_bin_col_name = BinningDataframeColumn.TIME_BIN.value
    bin_mid_col_name = BinningDataframeColumn.BIN_MIDPOINT.value
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value

    binned_dfs = []
    binned_dfs.append(data_dict[ExperimentDataKey.BINNED_NEUTRONS])
    binned_dfs.append(data_dict[ExperimentDataKey.BINNED_GAMMA])
    binned_reactor_data = data_dict.get(ExperimentDataKey.BINNED_REACTOR_DATA)
    if binned_reactor_data is not None:
        binned_reactor_param_dfs = binned_reactor_data.values()
        for binned_reactor_param_df in binned_reactor_param_dfs:
            binned_dfs.append(binned_reactor_param_df)
    merged_df = reduce(
        lambda df1, df2: pd.merge(
            df1, df2, how='left', on=[
                time_bin_col_name, bin_mid_col_name, bin_time_col_name
            ]
        ),
        binned_dfs
    )
    data_dict[ExperimentDataKey.ALL_BINNED_DATA] = merged_df

## Export and Display

In [ ]:
# clear old data from root, and make new analysis folder
# for exp_name in experiment_neutron_data.keys():
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     for file in root.iterdir():
#         if file.is_file() and not file.is_dir():
#             file.unlink()
#     analysis_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# add analysis setting file to experiment root and analysis folder
# for exp_name in experiment_neutron_data.keys():
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = f"{exp_name}_analysis_settings_{analysis_timestamp}.json"
#     with open(root / file_name, 'w') as root_settings_file:
#         json.dump(overall_settings, root_settings_file)
#     # with open(analysis_root / file_name, 'w') as analysis_settings_file:
#     #     json.dump(overall_settings, analysis_settings_file)
#     try:
#         shutil.copy(root / file_name, analysis_root / file_name)
#     except shutil.SameFileError:
#         pass

In [ ]:
# Export as CSV
# for exp_name, data_dict in experiment_neutron_data.items():
#     all_binned_data = data_dict[ExperimentDataKey.ALL_BINNED_DATA]
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = f"{exp_name}_data_{bin_length}s_bin.csv"
#     root_path = root / file_name
#     analysis_path = analysis_root / file_name
#     all_binned_data.to_csv(root_path, index=False)
#     try:
#         shutil.copy(root_path, analysis_path)
#     except shutil.SameFileError:
#         pass
#     print(f"Experiment {exp_name} saved to:")
#     print(f"    - {root_path}")
#     print(f"    - {analysis_path}")

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     binned_gamma_spectrum = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM]
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = (f"{exp_name}_gamma_spectrum_{bin_length}s_time_bin.csv")
#     root_path = root / file_name
#     analysis_path = analysis_root / file_name
#     binned_gamma_spectrum.to_csv(root_path)
#     try:
#         shutil.copy(root_path, analysis_path)
#     except shutil.SameFileError:
#         pass
#     print(f"Gamma spectrum for {exp_name} saved to:")
#     print(f"    - {root_path}")
#     print(f"    - {analysis_path}")

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
    
#     root = get_report_root(exp_name)
#     analysis_root = root / analysis_timestamp
#     file_name = f"{exp_name}_event_psd_energy.csv"
#     root_path = root / file_name
#     analysis_path = analysis_root / file_name
    
#     columns = [
#         calibrated_energy_column.value,
#         DetectorDataframeColumn.PSD.value,
#         DetectorDataframeColumn.NEW_N_CLASS.value
#     ]
#     headers = ['Energy (MeVee)', 'PSD', 'Is Neutron?']
#     psd_report.to_csv(
#         root_path,
#         index=False,
#         columns=columns,
#         header=headers
#     )
#     try:
#         shutil.copy(root_path, analysis_path)
#     except shutil.SameFileError:
#         pass
#     print(f"Event energy/PSD data for {exp_name} saved to:")
#     print(f"    - {root_path}")
#     print(f"    - {analysis_path}")

## Diagnostics Display

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
#     fig, ax = plot_scatter(
#         psd_report[calibrated_energy_column.value],
#         psd_report[DetectorDataframeColumn.PSD.value]
#     )
#     ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
#     ax.set_ylabel("PSD", fontsize=14)
    
#     ax.set_title(
#         f"{exp_name} PSD/Energy Graph",
#         ha='center',
#         fontsize=20
#     )
#     # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
#     from pathlib import Path
#     output_path = Path() / f"{exp_name} PSD graph.png"
#     print(str(output_path))
#     fig.savefig(str(output_path))
#     plt.show()

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
#     fom_results = data_dict[ExperimentDataKey.FOM_RESULTS]
#     x_bin_edges = helpers.get_midpoints_from_min_max_series(
#         fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MINIMUM.value],
#         fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MAXIMUM.value],
#         fom_results.index
#     )
#     gamma_mu = fom_results.mu1
#     gamma_sigma = fom_results.sigma1
#     neutron_sigma = fom_results.sigma2
    
#     window_sigma = 5*gamma_sigma
#     fom_sigma = 3*(gamma_sigma+neutron_sigma)
    
#     fig, ax = plot_scatter(
#         psd_report[calibrated_energy_column.value],
#         psd_report[DetectorDataframeColumn.PSD.value]
#     )
#     dot_size = 8
#     ax.scatter(
#         x_bin_edges,
#         window_sigma,
#         marker=".",
#         linewidths=0,
#         s=dot_size,
#         label="5 x gamma sigma"
#     )
#     ax.scatter(
#         x_bin_edges,
#         fom_sigma,
#         marker="o",
#         linewidths=0,
#         s=dot_size,
#         label="3 x sigma sum"
#     )
#     ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
#     ax.set_ylabel("PSD", fontsize=14)
#     # Add a title
#     ax.set_title(
#         f"{exp_name} PSD/Energy Graph",
#         ha='center',
#         fontsize=20
#     )
#     # output_path = get_report_root(exp_name) / f"{exp_name} PSD graph.png"
#     # fig.savefig(output_path)
#     plt.legend()
#     plt.show()

In [ ]:
# 4d Dwell Time and Count Rate Part 1 (Count rate plot) (2025-04-28)
### For named color options (and examples), check https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors
# (For more options, check https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors)
# (Any text option (in quotes) can be used)
for exp_name, data_dict in experiment_neutron_data.items():
    binned_neutrons = data_dict[ExperimentDataKey.ALL_BINNED_DATA]

    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
    
    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    rate_errors = binned_neutrons[n_error_col_name]

    # color_block_settings = []
    # add_color_blocks = helpers.get_input_with_default("Do you want to add color blocks? [y/N] >", "n", str)
    # if add_color_blocks.lower() == "y":
    #     color_blocks_count = helpers.get_input_required("How many color blocks do you want to add? (minimum 1) >", (1, None), int)
    #     for i in range(color_blocks_count):
    #         print(f"For block {i+1}:")
    #         block_start = helpers.get_input_required("Where should the block start (in minutes elapsed)? >", (0, None), int)
    #         block_end = helpers.get_input_required("Where should the block end (in minutes elapsed)? >", (block_start, None), int)
    #         block_color = input("What color should the block be? >")
    #         block_settings = {"start": block_start, "end": block_end, "color": block_color}
            
    #         block_needs_text = helpers.get_input_with_default("Does the block need bottom text? [y/N] >", "n", str)
    #         if block_needs_text.lower() == "y":
    #             block_text = input("Enter block text >")
    #             block_text_alignment = helpers.get_input_with_default(
    #                 "Choose alignment (left, center, right) or press Enter for default (center) >", "center", str
    #             )
    #             block_y_pos = helpers.get_input_required("Choose text's vertical position (in CPS) >", (0, 160), int)
    #             if block_text_alignment == "left":
    #                 block_text_position = (block_start, block_y_pos)
    #             elif block_text_alignment == "right":
    #                 block_text_position = (block_end, block_y_pos)
    #             else:
    #                 block_text_alignment = "center"
    #                 block_text_position = ((block_start + block_end) / 2, block_y_pos)
    #             block_settings["text"] = block_text
    #             block_settings["text_alignment"] = block_text_alignment
    #             block_settings["text_position"] = block_text_position
            
    #         color_block_settings.append(block_settings)
    
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    dot_size = 8
    
    ax.errorbar(
        zeroed_bins,
        rates,
        # yerr=rate_errors,
        # fmt=".",
        linestyle=':',
        # markersize=dot_size,
        # capsize=dot_size,
        markersize=0,
        color="black"
    )
    ax.bar(
        zeroed_bins,
        rates,
        # width=(zeroed_bins[1:] - zeroed_bins[:-1]),
        bin_length / 60,
        linewidth=1,
        edgecolor="black",
        facecolor="#00000000"
    )
    ax.set_xlabel("Time [minutes]", fontsize=14)  # Update x-axis label
    ax.set_ylabel("Neutron production rate [1/s]", fontsize=14)
    ax.tick_params(labelsize=12)
    # ax.set_ylim(0, 150)
    # ax.set_xlim(0, 145)
    
    # for block_settings in color_block_settings:
    #     ax.axvspan(block_settings["start"], block_settings["end"], color=block_settings["color"], alpha=0.2)
    #     block_text = block_settings.get("text")
    #     if block_text is not None:
    #         text_align = block_settings["text_alignment"]
    #         text_pos = block_settings["text_position"]
    #         ax.annotate(block_text, text_pos, horizontalalignment=text_align, fontsize=14)

    # # remove later
    # rate_lo = 13
    # rate_hi = 110
    # rel_diff = (rate_hi - rate_lo) / rate_hi
    # ax.set(xlim=(0, 150), ylim=(0, 140))
    # ax.hlines([rate_lo, rate_hi], -10, 160, linestyles="dashed", colors="black")
    # ax.axhspan(rate_lo, rate_hi, color="grey", alpha=0.1)
    # ax.annotate(f"{rel_diff:.0%} decrease", (20, (rate_lo + rate_hi) / 2), fontsize=14)

    # Add a title above the plot
    # fig.text(
    #     0.5,
    #     0.90,
    #     f"{exp_name} neutron count rate over time (Dwell time {bin_length}s)",
    #     ha='center',
    #     fontsize=20
    # )

    # # Save the plot
    # output_path = (
    #     # get_report_root(exp_name)
    #     Path()
    #     / f"{exp_name} Count rates {bin_length}s dwell.png"
    # )
    # fig.savefig(output_path)

    # Show the plot (optional)
    plt.show()

In [ ]:
# 3e 3D plot of neutron/gamma channels (AKA vaporwave island) (2025-04-24)
# cmap = plt.colormaps["nipy_spectral"]
cmap = plt.colormaps["viridis"]
figsize = (24, 24)
fontsize = 32
histo_res = 128
contour_res = 100
angle_elev = 30
angle_rot = -20
e_margin = 2
psd_margin = 0.002


for exp_name, data_dict in experiment_neutron_data.items():
    # xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
    # ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    # dz = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data["time_bin_energy_edges"]
    ye = exp_data["time_bin_psd_edges"]
    dz = exp_data["time_bin_histogram"]
    
    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    _xmids = (xe[:-1] + xe[1:]) / 2
    _ymids = (ye[:-1] + ye[1:]) / 2
    xmids, ymids = np.meshgrid(_xmids, _ymids)
    xmids, ymids = xmids.ravel(), ymids.ravel()
    # xmids = recalibrate_np(xmids)
    # ymids = recalibrate_np(ymids)
    if borders.left is None:
        within_left_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_left_border = xmids >= borders.left
    if borders.right is None:
        within_right_border = np.full_like(xmids, True, dtype=bool)
    else:
        within_right_border = xmids <= borders.right
    if borders.bottom is None:
        within_bottom_border = np.full_like(ymids, True, dtype=bool)
    else:
        bottom_border_psds = borders.bottom(xmids)
        within_bottom_border = ymids >= bottom_border_psds
    if borders.top is None:
        within_top_border = np.full_like(ymids, True, dtype=bool)
    else:
        top_border_psds = borders.top(xmids)
        within_top_border = ymids <= top_border_psds
    within_borders = (within_left_border &
                      within_right_border &
                      within_bottom_border &
                      within_top_border)

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin
    
    height_mask = dz > 1
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]
    within_borders = within_borders[height_mask]
    
    fig = plt.figure(figsize=figsize)
    ax = plt.axes(projection='3d', computed_zorder=False)

    # min_dz = np.min(dz)
    # min_dz = -2000
    min_dz = 0
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]
    mapped_colors = [
        bg_red if is_within_borders else color
        for is_within_borders, color in zip(within_borders, mapped_colors)
    ]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(x, y, z, dx, dy, dz,
             # color=color_alphas
             color=mapped_colors,
             shade=False,
             zsort="max",
             lw=0.2,
             ec="black"
            )

    border_e = np.linspace(lower_energy_bound, 3000, 200, dtype="float")
    bottom_border_psds = borders.bottom(border_e)
    top_border_psds = borders.top(border_e)
    border_zs = np.zeros_like(border_e, dtype="float")
    ax.plot(border_e, bottom_border_psds, zs=0, zdir='z', axlim_clip=True, lw=3, color="red", zorder=0)
    ax.plot(border_e, top_border_psds, zs=0, zdir='z', axlim_clip=True, lw=3, color="red", zorder=0)
    ax.fill_between(
        border_e, bottom_border_psds, border_zs,
        border_e, top_border_psds, border_zs,
        axlim_clip=True, color="red", alpha=0.5, zorder=0
    )

    bbox_params = {
        # "boxstyle": "round, pad=0.003, rounding_size=0.1",
        "fc": "white",
        "lw": 1,
        "fill": True,
        "alpha": 0.9
    }
    text_params = {
        "fontsize": fontsize-2,
        "bbox": bbox_params,
        "zorder": 5,
        "va": "center"
    }
    
    # gamma_text = ["Gamma", "channel"]
    # neutron_text = ["Neutron", "channel"]
    # # scaling = (400, 0.025)
    # scaling = (0.03, 280)
    # gamma_position = (4400, 0.135)
    # neutron_position = (4400, 0.35)
    # # mutation_aspect = 3500 / 0.5
    # # mutation_aspect = None
    # linespacing = 1
    # padding = 0.003
    # rounding_size = 0.1
    # font_properties = mpl.font_manager.FontProperties(weight="bold")

    # for text_list, position in [
    #     (gamma_text, gamma_position),
    #     (neutron_text, neutron_position)
    # ]:
    #     text_path = make_text_path(
    #         text_list,
    #         linespacing,
    #         # font_properties=font_properties
    #     )
    #     # bbox = make_bounding_box_for_text_path(
    #     #     text_path,
    #     #     padding,
    #     #     rounding_size,
    #     #     1,
    #     #     # mutation_aspect=mutation_aspect,
    #     #     **bbox_params
    #     # )
    #     transform = mpl.transforms.Affine2D()
    #     transform = transform.scale(*scaling)
    #     pos_from = get_bbox_center(text_path.get_extents(transform))
    #     x_translate, y_translate = get_translation_to(pos_from, position)
    #     transform = transform.translate(x_translate, y_translate)
    #     x_center, y_center = get_bbox_center(text_path.get_extents(transform))
    #     transform = transform.rotate_deg_around(x_center, y_center, 90)
    #     text_path = transform.transform_path(text_path)
    #     text_patch = convert_text_path_to_patch(text_path, 1)
    #     # patch_to_3d_plot_wall(bbox, ax, "z")
    #     patch_to_3d_plot_wall(text_patch, ax, "z")

    # arrow_width = 0.04
    # arrow_head_width = 0.07
    # arrow_head_length = 400
    # arrow_base_e = 3950
    # arrow_len_psd = 0
    # arrow_kwargs = {
    #     "width": arrow_width,
    #     "head_width": arrow_head_width,
    #     "head_length": arrow_head_length,
    #     "length_includes_head": True,
    #     "ec": "black",
    #     "fc": "none",
    #     "lw": 4
    # }
    # arrow_g = mpl.patches.FancyArrow(
    #     arrow_base_e, 0.135, -800, arrow_len_psd,
    #     **arrow_kwargs
    # )
    # patch_to_3d_plot_wall(arrow_g, ax, "z")
    # arrow_n = mpl.patches.FancyArrow(
    #     arrow_base_e, 0.35, -2550, arrow_len_psd,
    #     **arrow_kwargs
    # )
    # patch_to_3d_plot_wall(arrow_n, ax, "z")

    # ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
    ax.set_xlabel("Energy (ADC channel x1000)", fontsize=fontsize)
    ax.set_ylabel("PSD", fontsize=fontsize)
    ax.set_zlabel("Counts", fontsize=fontsize)
    ax.set_xlim(0, 3000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 200)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    # ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize)
    ax.tick_params(pad=0)
    ax.tick_params(axis="x", pad=8)
    ax.tick_params(axis="y", pad=4)
    # ax.tick_params(axis="z", pad=-2)
    # for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
    #     axis3d.labelpad = 40
    ax.xaxis.labelpad = 48
    ax.yaxis.labelpad = 40
    ax.zaxis.labelpad = 44
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))
    ax.set_box_aspect(None, zoom=0.85)

In [ ]:
# # import matplotlib as mpl
# HISTOGRAM_RES = 1024
# COUNT_LIMIT = 20

# for exp_name, data_dict in experiment_neutron_data.items():
#     psd_report = data_dict[ExperimentDataKey.PSD_REPORT]
#     borders = data_dict[ExperimentDataKey.BORDERS]
    
#     fig, ax = plot_classification(
#         psd_report,
#         borders,
#         exp_name,
#         DetectorDataframeColumn.NEW_N_CLASS,
#         calibrated_energy_column,
#         count_limit=COUNT_LIMIT,
#         colormap_name="seismic"
#     )

#     output_path = (
#         # get_report_root(exp_name)
#         Path()
#         / f"{exp_name} Neutron Classification.png"
#     )
#     fig.savefig(output_path)

#     plt.show()

In [ ]:
# fom_to_sigma = 2 * sqrt(2 * log(2))

# for exp_name, data_dict in experiment_neutron_data.items():
#     fom_results = data_dict[ExperimentDataKey.FOM_RESULTS]
#     slice_mid_energy = helpers.get_midpoints_from_min_max_series(
#         fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MINIMUM.value],
#         fom_results[SliceFitDataframeColumn.SLICE_ENERGY_MAXIMUM.value],
#         fom_results.index
#     ).to_numpy(copy=True)

#     fig, ax = plt.subplots(figsize=(8, 8))
#     ax.plot(
#         slice_mid_energy,
#         fom_results[SliceFitDataframeColumn.FOM.value]
#     )
#     ax.set_xlabel("Energy [MeVee]", fontsize=14)  # Update x-axis label
#     ax.set_ylabel("FOM", fontsize=14)
#     # Add a title
#     ax.set_title(
#         f"{exp_name} FOM Graph",
#         ha='center',
#         fontsize=20
#     )
#     ax.hlines(1.27, slice_mid_energy[0], slice_mid_energy[-1], "r", ls="--")

#     output_path = (
#         # get_report_root(exp_name)
#         Path()
#         / f"{exp_name} Energy vs FOM.png"
#     )
#     fig.savefig(output_path)
#     plt.show()

In [ ]:
# # histogram contour plot (vaporwave island)
# cmap = plt.colormaps["nipy_spectral"]
# figsize = (12, 12)
# fontsize = 16
# histo_res = 128
# contour_res = 100
# angle_elev = 30
# angle_rot = -60

# for exp_name, data_dict in experiment_neutron_data.items():
#     Z = data_dict[ExperimentDataKey.PSD_HISTOGRAM]
#     xe = data_dict[ExperimentDataKey.HISTOGRAM_X_EDGES]
#     ye = data_dict[ExperimentDataKey.HISTOGRAM_Y_EDGES]
#     fig = plt.figure(figsize=figsize)
#     ax = plt.axes(projection='3d')
#     x, y = np.meshgrid(xe[:-1], ye[:-1])

#     ax.view_init(angle_elev, angle_rot)
#     ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)
#     ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
#     ax.set_ylabel("PSD", fontsize=fontsize)
#     ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
#     ax.set_zlabel("Counts", fontsize=fontsize)

#     output_path = (
#         # get_report_root(exp_name)
#         Path()
#         / f"{exp_name} PSD 3D Histogram.png"
#     )
#     fig.savefig(output_path)

#     plt.show()

In [ ]:
# cmap = plt.colormaps["nipy_spectral"]
# figsize = (12, 12)
# fontsize = 16
# contour_res = 50
# angle_elev = 30
# angle_rot = -60
# ls = LightSource(270, 45)

# for exp_name, data_dict in experiment_neutron_data.items():
#     Z = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM].to_numpy()
#     xe = data_dict[ExperimentDataKey.GAMMA_ENERGY_BIN_EDGES]
#     # TODO find x edge closest to high cutoff (0.5)
#     time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
#     start_time = time_bins[0]
#     ye = (time_bins - start_time).total_seconds() / 3600
#     xmid = (xe[1:] + xe[:-1]) / 2
#     ymid = (ye[1:] + ye[:-1]) / 2
#     # xmid = xe[:-1]
#     # ymid = ye[:-1]

#     fig = plt.figure(figsize=figsize)
#     ax = plt.axes(projection='3d')
#     x, y = np.meshgrid(xmid, ymid)

#     ax.view_init(angle_elev, angle_rot)
#     rgb = ls.shade(Z, cmap=cmap, blend_mode='soft')
#     # ax.contour3D(x, y, Z, contour_res, cmap=cmap)
#     ax.plot_surface(
#         x, y, Z,
#         rstride=1, cstride=1,
#         facecolors=rgb, linewidth=0,
#         antialiased=False, shade=False)
#     ax.set_title(f"{exp_name} PSD/Energy 3D Histogram", fontsize=fontsize+4)
#     ax.set_ylabel("Time (hours)", fontsize=fontsize)
#     ax.set_xlabel("Energy (MeVee)", fontsize=fontsize)
#     ax.set_zlabel("Counts", fontsize=fontsize)

#     output_path = (
#         # get_report_root(exp_name)
#         Path()
#         / f"{exp_name} Gamma Spectrum 3D Histogram.png"
#     )
#     fig.savefig(output_path)

#     plt.show()

In [ ]:
# for exp_name, data_dict in experiment_neutron_data.items():
#     gamma_energy_spectrum_df = data_dict[ExperimentDataKey.GAMMA_ENERGY_SPECTRUM]
#     do_spectrum = input(f"Do you want a gamma spectrum for {exp_name}? [y/n] ")
#     if do_spectrum.lower() not in ["y", "yes"]:
#         continue
#     elapsed_str = input(
#         "At what time (elapsed hours) do you want to get the spectrum? ")
#     try:
#         spectrum_time_elapsed = float(elapsed_str)
#     except ValueError:
#         print(f"The value {elapsed_str} was not a valid decimal number")
#         continue
#     time_bins = data_dict[ExperimentDataKey.TIME_BIN_EDGES]
#     start_time = time_bins[0]
#     spectrum_timestamp = start_time + timedelta(hours=spectrum_time_elapsed)

#     matching_intervals = [
#         interval for interval
#         in gamma_energy_spectrum_df.index.categories
#         if spectrum_timestamp in interval
#     ]
#     if len(matching_intervals) == 0:
#         print((
#             f"The given time ({spectrum_time_elapsed} hrs) " +
#             "has no match in this experiment"
#         ))
#         continue
#     matching_interval = matching_intervals[0]
#     gamma_test_spectrum = gamma_energy_spectrum_df[
#         gamma_energy_spectrum_df.index == matching_interval]

#     gamma_energy_bins = gamma_test_spectrum.T.index.to_series()
#     midpoints = gamma_energy_bins.apply(lambda x: x.mid)
#     fig, ax = plt.subplots(figsize=(8, 8))
#     ax.scatter(midpoints, gamma_test_spectrum.T, s=1)
#     ax.set_title(
#         f"Gamma Energy Spectrum - {exp_name} @ {spectrum_time_elapsed} hrs")
#     ax.set_xlabel("Energy (MeVee)")
#     ax.set_ylabel("Count")

#     plt.show()

## Done!

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

### Post-Completion

In [ ]:
list(experiment_neutron_data.keys())

In [ ]:
labels = {
    "ID-479": "June - 2 hours",
    "TB-46": "September - 0.5 hours (with KOH)",
    # "TB-47": "October - 24 hours"
    "TB-47": "24 hours background at UBC"
}

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
dot_size = 8

for exp_id, data_dict in experiment_neutron_data.items():
# data_dict = experiment_neutron_data["ID-479"]
    binned_neutrons = data_dict[ExperimentDataKey.ALL_BINNED_DATA]
    
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    n_rate_col_name = BinningDataframeColumn.NEUTRON_RATE.value
    n_error_col_name = BinningDataframeColumn.NEUTRON_RATE_ERROR.value
    
    zeroed_bins = binned_neutrons[bin_time_col_name] / 60
    rates = binned_neutrons[n_rate_col_name]
    rate_errors = binned_neutrons[n_error_col_name]
    
    hist, edges = np.histogram(rates, bins="fd")
    midpoints = (edges[1:] + edges[:-1])/2
    
    hist_mask = hist == 0
    hist = hist[~hist_mask]
    midpoints = midpoints[~hist_mask]
    data_dict["rate_histo"] = hist
    data_dict["rate_bin_mids"] = midpoints
    data_dict["rate_bin_edges"] = edges
    
    ax.scatter(midpoints, hist, label=labels.get(exp_id, "???"))
    ax.set(xlabel="Neutron rate (counts per second)", ylabel="N")
ax.legend()
plt.show()

In [ ]:
from scipy.optimize import curve_fit
from data_processing.processing.figure_of_merit import gaussian

for exp_id, data_dict in experiment_neutron_data.items():
    edges = data_dict["rate_bin_edges"]
    midpoints = data_dict["rate_bin_mids"]
    hist = data_dict["rate_histo"]
    
    guess_A = max(hist)
    max_idx = np.where(hist == guess_A)[0][0]
    guess_mu = midpoints[max_idx]
    guess_sigma = (edges[-1] - edges[0]) / 4
    
    params, cov = curve_fit(gaussian, midpoints, hist, p0=(guess_mu, guess_sigma, guess_A))
    print(exp_id)
    print(params)
    
    data_dict["hist_fit_params"] = params
    data_dict["hist_fit_cov"] = cov

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
dot_size = 8

for exp_id, data_dict in experiment_neutron_data.items():
    edges = data_dict["rate_bin_edges"]
    params = data_dict["hist_fit_params"]
    midpoints = data_dict["rate_bin_mids"]
    hist = data_dict["rate_histo"]

    mu, sigma, A = params
    
    x = np.linspace(edges[0], edges[-1], num=100)
    y = gaussian(x, *params)
    
    ax.scatter(midpoints, hist, label=labels.get(exp_id, "???"), color="black")
    ax.plot(x, y, color="black")
    # ax.vlines(mu, 0, A, color="black", linestyle="dashed")
    # ax.hlines(A / 2, mu-sigma, mu+sigma, color="black", linestyle="dashed")
    # ax.text(mu, A, f"Mean: {mu:.2f} cps", horizontalalignment="center", bbox={"boxstyle": "round", "color": "#FFFFFFD0"})
    # ax.text(mu, A/2+5, f"SD: {sigma:.2f} cps", horizontalalignment="center", bbox={"boxstyle": "round", "color": "#FFFFFFD0"})

ax.set(xlabel="Neutron production rate (1/s)", ylabel="Counts")
ax.legend()
plt.show()

In [ ]:
experiment_neutron_data["ID-479"][ExperimentDataKey.PSD_REPORT]["TIMETAG"].max() / 10e12 / 60 / 60

In [ ]:
experiment_neutron_data["TB-46"][ExperimentDataKey.PSD_REPORT]["TIMETAG"].max() / 10e12 / 60 / 60